In [38]:
import shutil
from pathlib import Path

PROXY_DOWNLOADS = Path("proxy_downloads")
SUBMISSIONS_DIR = Path("starter/proxylab-handout/submissions")
SUBMISSIONS_DIR.mkdir(exist_ok=True)
ctr = 1
NUM_FILES = 50

for proxy_file in PROXY_DOWNLOADS.rglob("proxy.c"):
    if ctr > NUM_FILES:
        break
    repo_name = proxy_file.parent.name
    dest = SUBMISSIONS_DIR / f"{repo_name}.c"
    shutil.copy(proxy_file, dest)

    repo_sub = SUBMISSIONS_DIR / repo_name
    repo_sub.mkdir(exist_ok=True)

    for extra in ("cache.c", "cache.h"):
        src = proxy_file.parent / extra
        if src.exists():
            shutil.copy(src, repo_sub / extra)
    ctr += 1

In [39]:
!chmod +x public_code_cleaning/prescreen.sh
!bash public_code_cleaning/prescreen.sh

Screening 50 submissions...

[FAIL] Compilation Error for 5antelope__CSAPP.c:
/work/starter/proxylab-handout/submissions/5antelope__CSAPP.c: In function ‘main’:
/work/starter/proxylab-handout/submissions/5antelope__CSAPP.c:56:39: error: passing argument 1 of ‘Open_listenfd’ makes pointer from integer without a cast [-Wint-conversion]
   56 |         if ((listenfd = Open_listenfd(port)) < 0) {
      |                                       ^~~~
      |                                       |
      |                                       int
In file included from /work/starter/proxylab-handout/submissions/5antelope__CSAPP.c:4:
/work/starter/proxylab-handout/csapp.h:195:25: note: expected ‘char *’ but argument is of type ‘int’
  195 | int Open_listenfd(char *port);
      |                   ~~~~~~^~~~
/work/starter/proxylab-handout/submissions/5antelope__CSAPP.c: In function ‘routine’:
/work/starter/proxylab-handout/submissions/5antelope__CSAPP.c:91:9: warning: variable ‘rp’ set but not us

In [35]:
from prompt_engineering.utils import MutationType

In [ ]:
from pathlib import Path
from prompt_engineering.regex import run_step1

HELPER_FILES = {"csapp.c", "port-for-user.c", "cache.c"}
SUBMISSIONS_DIR = Path("starter/proxylab-handout/submissions").resolve()
folder_path = Path(SUBMISSIONS_DIR)
data = "public_testset"

for mutation in MutationType:
    failed = []
    all_chunks = []
    for file_path in folder_path.iterdir():
        if file_path.is_file() and file_path.suffix == ".c":
            if file_path.name in HELPER_FILES:
                continue

            chunk = run_step1(file_path, mutation)
            if chunk:
                all_chunks.append({"id": file_path.name, "chunk": chunk})
            else:
                failed.append(file_path.name)

    with open(f"chunks/{data}_{mutation}.py", "w") as f:
        f.write("from prompt_engineering.utils import Chunk\n\n")
        f.write("chunks = [\n")
        for c in all_chunks:
            f.write(f"\t{{'id': {c['id']!r}, 'chunk': {c['chunk']!r}}},\n")
        f.write("]\n")

    print(f"Mutation: {mutation}, Num Succeeded: {len(all_chunks)}, Num Failed: {len(failed)}")
    # print("Failed: ", list(sorted(failed)))

Mutation: protocol, Num Succeeded: 102, Num Failed: 96
Mutation: robustio, Num Succeeded: 57, Num Failed: 141
Mutation: buflen, Num Succeeded: 28, Num Failed: 170
Mutation: parsing, Num Succeeded: 146, Num Failed: 52
